In [27]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO


def extract_bls_tables():
    """
    Attempt to extract tables from BLS Women's Databook
    """
    url = "https://www.bls.gov/opub/reports/womens-databook/2021/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse HTML and extract tables
        soup = BeautifulSoup(response.content, "html.parser")
        tables = soup.find_all("table")

        # Extract specific tables (11-14)
        target_tables = []
        for i, table in enumerate(tables):
            if i in [11, 14]:
                target_tables.append(table)

        return target_tables

    except Exception as e:
        print(f"Error: {e}")
        return None

In [28]:
import pandas as pd
from bs4 import BeautifulSoup
import re


def add_hierarchy_context(df):
    """
    Add parent industry information to maintain hierarchy context.

    Args:
        df (pd.DataFrame): DataFrame with basic hierarchy information

    Returns:
        pd.DataFrame: DataFrame with parent industry information added
    """

    # Initialize parent columns
    df["parent_level_0"] = None
    df["parent_level_1"] = None
    df["parent_level_2"] = None
    df["parent_level_3"] = None

    # Track current parents at each level
    current_parents = {}

    for idx, row in df.iterrows():
        level = row["hierarchy_level"]
        industry = row["industry_name"]

        # Update current parent for this level
        current_parents[level] = industry

        # Clear deeper levels
        levels_to_clear = [l for l in current_parents.keys() if l > level]
        for l in levels_to_clear:
            del current_parents[l]

        # Set parent columns
        for parent_level in range(4):
            if parent_level in current_parents and parent_level < level:
                df.at[idx, f"parent_level_{parent_level}"] = current_parents[
                    parent_level
                ]

    return df


def extract_hierarchical_table_data(html_content):
    """
    Extract hierarchical data from the HTML table into a pandas DataFrame.

    Args:
        html_content: A tag containing the table

    Returns:
        pd.DataFrame: DataFrame with hierarchy information preserved
    """
    soup = html_content

    # Find the tbody element
    tbody = soup.find("tbody")

    # Extract data from each row
    data = []

    for row in tbody.find_all("tr"):
        # Extract the header cell (industry name)
        th = row.find("th")
        if th:
            # Get the paragraph element with the class indicating hierarchy level
            p_element = th.find("p")
            if p_element:
                # Extract hierarchy level from class
                class_list = p_element.get("class", [])
                hierarchy_level = None
                for cls in class_list:
                    if cls.startswith("sub"):
                        hierarchy_level = int(cls[3:])  # Extract number from 'subX'
                        break

                # If no sub class found, it's level 0
                if hierarchy_level is None:
                    hierarchy_level = 0

                # Get the industry name
                industry_name = p_element.get_text(strip=True)

                # Extract data from td elements
                td_elements = row.find_all("td")
                employment = None
                percentage = None

                if len(td_elements) >= 2:
                    employment_text = td_elements[0].get_text(strip=True)
                    percentage_text = td_elements[1].get_text(strip=True)

                    # Clean and convert employment (remove commas)
                    if employment_text and employment_text != "-":
                        employment = int(employment_text.replace(",", ""))

                    # Clean and convert percentage
                    if percentage_text and percentage_text != "-":
                        percentage = float(percentage_text)

                # Add to data list
                data.append(
                    {
                        "industry_name": industry_name,
                        "hierarchy_level": hierarchy_level,
                        "employment": employment,
                        "percentage_female": percentage,
                    }
                )

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add additional hierarchy information
    df = add_hierarchy_context(df)

    return df

In [29]:
tables = extract_bls_tables()

In [31]:
industry_gender_share = extract_hierarchical_table_data(tables[1])
industry_gender_share.head()

,industry_name,hierarchy_level,employment,percentage_female,parent_level_0,parent_level_1,parent_level_2,parent_level_3
0,"Total, 16 years and older",0,147795,46.8,None,None,None,None
1,"Agriculture, forestry, fishing, and hunting",0,2349,27.7,None,None,None,None
2,Crop production,1,1186,28.9,"Agriculture, forestry, fishing, and hunting",None,None,None
3,Animal production and aquaculture,1,832,28.1,"Agriculture, forestry, fishing, and hunting",None,None,None
4,Support activities for agriculture and forestry,1,145,33.9,"Agriculture, forestry, fishing, and hunting",None,None,None


In [33]:
occupation_gender_share = extract_hierarchical_table_data(tables[0])
occupation_gender_share.head()

,industry_name,hierarchy_level,employment,percentage_female,parent_level_0,parent_level_1,parent_level_2,parent_level_3
0,"Total, 16 years and older",0,147795,46.8,None,None,None,None
1,"Management, professional, and related occupations",0,63644,51.7,None,None,None,None
2,"Management, business, and financial operations...",1,27143,44.6,"Management, professional, and related occupations",None,None,None
3,Management occupations,2,18564,40.4,"Management, professional, and related occupations","Management, business, and financial operations...",None,None
4,Chief executives,3,1669,29.3,"Management, professional, and related occupations","Management, business, and financial operations...",Management occupations,None


In [34]:
occupation_gender_share.to_csv(
    "../data/derived/occupation_gender_share.csv", index=False
)
industry_gender_share.to_csv("../data/derived/industry_gender_share.csv", index=False)